# Software profesional en Acústica 2025-26 (M2i)

*This notebook was adapted from Chapter 1 of [The FEniCS Tutorial Volume I](https://fenicsproject.org/pub/tutorial/sphinx1/) by Hans Petter Langtangen and Anders Logg, released under CC Attribution 4.0 license. It has been created by Xiangmin Jiao (University of Stony Brook University) and it is available in the repository [Unifem/FEniCS-note](https://github.com/unifem/fenics-notes). The FEniCS implementation has been replaced with NGSolve.*

This Jupyter Notebook has been designed to be run in [Google Colab](https://colab.research.google.com/). With this purpose the first cell install [NGSolve](https://ngsolve.org/) related packages in a clean machine (if they have not been previously installed). Typically, this installation takes less than 2 minutes.

In [19]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# The equations of vibrations in fluid-structure interaction (pressure-displacement formulation)

Vibrations in linear elasticity is the study of how solid objects are responding to a mechanical vibration and become 
internally stressed due to prescribed time-harmonic loading conditions. It is an important problem
in modern engineering. Its corresponding PDE is a generalization of the
Helmholtz equation, and it is among one of the most popular PDEs in 
engineering. We now study its variational formulation of a fluid-structure interaction and how to solve
this problem using NGSolve in 2D.

## PDE problem

The time-harmonic equation governing vibrations of a fluid-structure problem involving a elastic solid structure $\Omega_S$ and a fluid domain $\Omega_F$ can be written as

\begin{align}
\tag{1}
&-\omega^2\rho_S\boldsymbol{u}_S -\boldsymbol{\nabla}\cdot\boldsymbol{\sigma} = 0\hbox{ in }\Omega_S,\\
&-\omega^2\frac{1}{\rho_Fc^2}p_F -\frac{1}{\rho_F}\Delta p_F = 0\hbox{ in }\Omega_F,\tag{2}
\end{align}

where $\boldsymbol{\sigma}$ is the *stress tensor*, $c$ is the sound speed in the fluid domain, $\boldsymbol{u}_S$ is the displacement field in the solid, $p_F$ is the acoustic pressure in the fluid domain,
$\rho_S$ and $\rho_F$ are the *mass density* of the solid and fluid domain, and $\omega$ the angular frequency. For isotropic materials, the stress tensor is further related to the deformation by 
the following two equations:
\begin{align}
\boldsymbol{\sigma} &= \lambda\,\hbox{tr}\,(\boldsymbol{\varepsilon}) \boldsymbol{I} + 2\mu\boldsymbol{\varepsilon},
\tag{3}\\
\boldsymbol{\varepsilon} &= \frac{1}{2}\left(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top}\right),
\tag{4}
\end{align}
where $\boldsymbol{\varepsilon}$ is the *symmetric strain-rate tensor* (symmetric gradient), 
and $\boldsymbol{u}_S$ is the *displacement vector field*, $\boldsymbol{I}$ denotes the *identity tensor*, 
$\mathrm{tr}$ denotes the *trace operator* on a tensor, and $\lambda$ and $\mu$ 
are material properties known as *Lamé's elasticity parameters*.

We can combine (3) and (4) to obtain
\begin{equation}
\tag{5}
\boldsymbol{\boldsymbol{\sigma}} = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u}_S)\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top})
\end{equation}

## Variational formulation

The variational formulation of
(1)-(4)
consists of forming the inner product of
(1)-(2) and an *scalar* and a *vector* test functions
$(p_F,\boldsymbol{v}_S)\in \hat{V}$, where $\hat{V}$ is a mixed scalar and vector-valued test function space in $\Omega_F$ and $\Omega_S$ such that the following conditions are satisfied
\begin{align}
\frac{1}{\omega^2\rho}\nabla p_{F}\cdot \mathbf{n} = \boldsymbol{v}_S\cdot \mathbf{n},\\
\boldsymbol{\sigma}(\boldsymbol{u}_S)\mathbf{n} = -p_{F},
\end{align}

on the coupling boundary $\Gamma_I=\partial\Omega_F\cap\partial\Omega_F$. So, integrating over the domain $\Omega_F\cup\Omega_S$ and taking into account the symmetry of the tensor of elasticity, it holds

\begin{align}
-\omega^2\int_{\Omega_S} \rho_S\boldsymbol{u}_S\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_S} \boldsymbol{\sigma}(\boldsymbol{u}_S) : \boldsymbol{\epsilon}(\boldsymbol{v}_S)\ \mathrm{d}\boldsymbol{x} 
 -\omega^2\int_{\Omega_F} \frac{1}{\rho_F c^2}p_F q_F\ \mathrm{d}\boldsymbol{x}\\
 -\int_{\Gamma_I}p_{F}\boldsymbol{v}_S\cdot\mathbf{n}\ \mathrm{d}\boldsymbol{s} 
 -\omega^2\int_{\Gamma_I}q_{F}\boldsymbol{u}_S\cdot\mathbf{n}\ \mathrm{d}\boldsymbol{s}
 + \int_{\Omega_F} \frac{1}{\rho_F}\nabla p_F\cdot \nabla q_F \mathrm{d}\boldsymbol{x} 
 = \int_{\Gamma_T} \boldsymbol{T}\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{s}
\tag{8}
\end{align}

for all $(q_F,\boldsymbol{v}_S)\in \hat{V}$ such that $\boldsymbol{u}_S=\mathbf{0}$ on the campled boundary $\Gamma_C=\{\boldsymbol{x}\in\partial\Omega_{S}: x_0=0\}$ and the traction boundary $\Gamma_T=\{\boldsymbol{x}\in\partial\Omega_{S}: x_0=L\}$. In addition,
$\boldsymbol{\epsilon}(\boldsymbol{v})$ is the symmetric part of $\boldsymbol{\nabla} \boldsymbol{v}$.

### Enforcing boundary conditions

Now let us consider how to enforce boundary conditions. 
For Dirichlet boundaries, we will enforce boundary-conditions strongly.
For these points, no test functions are associated with the Dirichlet nodes.

For traction boundary conditions, we will enforce the boundary condition
weakly using the variational form (8).
Similar to the Helmholtz equation, we require their corresponding test
functions $\boldsymbol{v}$ vanish along $\partial \Omega$ for interior points.
Then, the boundary integral above has no effects for points on
$\partial\Omega\setminus\partial\Omega_T$.

### Summary of variational form
In summary, the variational problem is to find $\boldsymbol{u}$ in a vector function space $\hat{V}$ such that
\begin{equation}
a((p_F,\boldsymbol{u}_S),(q_F,\boldsymbol{v}_S)) = L((q_F,\boldsymbol{v}_S))\quad\forall (q_F,\boldsymbol{v}_S)\in\hat{V},
\end{equation}
where 

\begin{align}
a((p_F,\boldsymbol{u}_S),(q_F,\boldsymbol{v}_S)) =& -\omega^2\int_{\Omega_S} \rho_S\boldsymbol{u}_S\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_S} \boldsymbol{\sigma}(\boldsymbol{u}_S) : \boldsymbol{\epsilon}(\boldsymbol{v}_S)\ \mathrm{d}\boldsymbol{x} \\
 &-\omega^2\int_{\Omega_F} \frac{1}{\rho_F c^2}p_F q_F\ \mathrm{d}\boldsymbol{x}
 -\int_{\Gamma_I}p_{F}\boldsymbol{v}_S\cdot\mathbf{n}\ \mathrm{d}\boldsymbol{s}\\ 
 &-\omega^2\int_{\Gamma_I}q_{F}\boldsymbol{u}_S\cdot\mathbf{n}\ \mathrm{d}\boldsymbol{s}
 + \int_{\Omega_F} \frac{1}{\rho_F}\nabla p_F\cdot \nabla q_F \mathrm{d}\boldsymbol{x}  
\end{align}
and 
\begin{equation}
\boldsymbol{\sigma}(\boldsymbol{u}_S) = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u}_S)\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top}).\\
\end{equation}

## NGSolve implementation

To demonstrate the implementation, we will model a clamped beam deformed under a time-harmonic surface force on the opposite free cross-section surface. This can be modeled by setting $\boldsymbol{T}=(0,0,1)$ on that boundary $\Gamma_T$. The solid structure is box-shaped with length and width $L$, whereas the fluid domain is an interior box-shaped domain with length and width $W<L$. We
set $\boldsymbol{u}=(0,0,0)$ at the clamped end, $x=0$. The rest of the lateral boundary is
traction free; that is, we set $\boldsymbol{T} = 0$. Therefore,
$$L((q_F,\boldsymbol{v}_S)) = \int_{\Gamma_T} \boldsymbol{T}\cdot \boldsymbol{v}_S \mathrm{d}\boldsymbol{s}$$
for this problem.

### Import packages

We start by importing NGSolve.

In [20]:
import numpy as np
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

### Generate the mesh and function spaces

We use Netgen to create a geometry with two named material subdomains (solid and fluid) and tagged boundaries.

In [21]:
# Geometry
L = 1.0

# Primary geometric objects
domain_solid = Rectangle(L, L).Face()
domain_fluid = MoveTo(L/4, L/4).Rectangle(L/2, L/2).Face()

# Domain and boundary tags for the upper part
domain_solid.edges.Min(Y).name = "clamped"
domain_solid.edges.Min(Y).col = (1, 0, 0) # red
domain_solid.edges.Max(Y).name = "top"
domain_solid.edges.Max(Y).col = (0, 0, 1) # blue
domain_solid.edges.Min(X).name = "free"
domain_solid.edges.Min(X).col = (0, 0, 1) # blue
domain_solid.edges.Max(X).name = "free"
domain_solid.edges.Max(X).col = (0, 0, 1) # blue

# Domain and boundary tags for the lower part
domain_fluid.edges.name = "interface"
domain_fluid.edges.col = (1, 1, 0) # yellow
domain_fluid.faces.name = "fluid"
domain_fluid.faces.col = (0, 0, 1)  # Blue color for faces
domain_fluid.mat("fluid")

# Subtract the fluid domain from the solid domain to create a non-overlapping geometry
domain_solid  = domain_solid  - domain_fluid
domain_solid.faces.name = "solid"
domain_solid.faces.col = (0, 1, 0)  # Green color for faces
domain_solid.mat("solid")

# Create the domain
domain = Glue([domain_solid, domain_fluid])

## Each edge is colored
Draw(domain, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [22]:
# Define the mesh
h_size = 0.05
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=h_size, quad_dominated=False))

print("Number of elements:", mesh.ne)
print("Materials:", mesh.GetMaterials())
print("Boundaries:", mesh.GetBoundaries())

# Plot mesh
Draw(mesh, height="3vh")

Number of elements: 938
Materials: ('solid', 'fluid')
Boundaries: ('clamped', 'free', 'free', 'top', 'interface', 'interface', 'interface', 'interface')


WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Define the mixed finite element space

For the pressure-displacement formulation we use a **mixed space**: a vector $H^1$ space (order 2) for the solid displacement $\boldsymbol{u}_S$, and a scalar $H^1$ space (order 1) for the acoustic pressure $p_F$. In NGSolve these are combined with `FESpace`.

In [23]:
# Individual spaces
V_u = VectorH1(mesh, order=2, dirichlet="clamped", complex=True, definedon="solid")  # displacement (solid)
V_p = H1(mesh,       order=2, complex=True, definedon="fluid")                        # pressure    (fluid)

# Mixed (product) space: W = V_u x V_p
W = V_u * V_p
print("Total DOFs:", W.ndof)

Total DOFs: 4139


### Define the variational problem

With `(u, p)` as trial functions and `(v, q)` as test functions, we assemble the bilinear form from (8) using domain- and boundary-restricted measures.

In [24]:
# Physical parameters
omega_val = 2 * np.pi * 1.0
rho_fluid = 0.2
c_val     = 1.0
rho       = 1.0
beta      = 1.25
lambda_   = beta
mu_val    = 1.0

# Strain and stress for the solid
def epsilon(w):
    return Sym(Grad(w))

def sigma(w):
    return lambda_ * Trace(Grad(w)) * Id(2) + 2 * mu_val * epsilon(w)

# Trial and test pairs from the mixed space
(u, p) = W.TrialFunction()
(v, q) = W.TestFunction()

# Domain-restricted measures
dx_solid = dx(definedon=mesh.Materials("solid"))
dx_fluid = dx(definedon=mesh.Materials("fluid"))

# Interface measure (interior facets tagged 'interface')
# In NGSolve, dS integrates over interior facets; we restrict to 'interface'
dI = ds(definedon=mesh.Boundaries("interface"))

# Normal vector on facets
n = specialcf.normal(mesh.dim)

# Bilinear form
a = BilinearForm(W)

# Solid block
a += (-omega_val**2 * rho * InnerProduct(u, v))             * dx_solid
a += InnerProduct(sigma(u), epsilon(v))                     * dx_solid

# Fluid block
a += (-omega_val**2 / (rho_fluid * c_val**2) * p * q)       * dx_fluid
a += (1.0 / rho_fluid) * InnerProduct(Grad(p), Grad(q))     * dx_fluid

# Fluid-structure coupling terms on the interface
# -∫_Γ p v·n ds  (pressure loading on the solid test function)
a += (-p * InnerProduct(v, n))                              * dI
# -ω² ∫_Γ q u·n ds  (inertial coupling)
a += (-omega_val**2 * q * InnerProduct(u, n))               * dI

a.Assemble()

# Linear form — traction (0, 1) on the right boundary
f_vec = CF((0, 1*np.exp(1j*np.pi/4)))  # Complex traction vector
L = LinearForm(W)
L += InnerProduct(f_vec, v) * ds(definedon=mesh.Boundaries("top"))
L.Assemble()

print("Bilinear and linear forms assembled.")

Bilinear and linear forms assembled.


### Solve the variational problem

In [25]:
# Compute solution
gfu = GridFunction(W)

# Apply the Dirichlet condition on the displacement component only
gfu.vec.data = a.mat.Inverse(W.FreeDofs()) * L.vec

# Extract displacement and pressure
u_sol, p_sol = gfu.components
print("System solved.")

System solved.


### Plot the solution

In [27]:
# First component of the solution (x-displacement)
Draw(u_sol[0], mesh.Materials("solid"), animate_complex=True, height="3vh", min=-0.075, max=0.075)
# Second component of the solution (y-displacement)
Draw(u_sol[1], mesh.Materials("solid"), animate_complex=True, height="3vh", min=-0.1, max=0.1)
# Total displacement magnitude
Draw(u_sol, mesh.Materials("solid"), deformation=0.5*u_sol.real, height="3vh", min=-0.075, max=0.075)
# Pressure field in the fluid domain
Draw(p_sol, mesh.Materials("fluid"), animate_complex=True, height="3vh", min=-0.1, max=0.1)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene